# GeoLift-S3 Lite + Teacher KD — augmentation ladder A1/A2/A3

Notebook này giữ nguyên architecture, teacher cache, loss, split 1.600/400 và LR trajectory của run Teacher-KD 20 epoch. Mỗi lần chạy chọn đúng một stage bằng `AUGMENTATION_STAGE`; ba stage phải chạy tuần tự để tạo ablation có thể quy attribution.

| Stage | Augmentation | Control trực tiếp |
|---|---|---|
| A1 | Horizontal flip, `p=0.5` | Teacher-KD plain |
| A2 | A1 + sparse dropout `U(0,0.2)` | A1 |
| A3 | A2 + isotropic zoom/crop `U(1.0,1.1)` | A2 |

Spatial transform được áp dụng đồng bộ lên RGB, sparse/GT, `D_cm/C_cm`, `R_G/C_G` và camera intrinsics. Sparse dropout chỉ sửa student sparse input/mask; GT và teacher targets giữ nguyên. Validation không có augmentation.

Success criteria: contract tests pass, visualization giữ teacher alignment, run có đủ 20 epoch, best checkpoint được chọn bằng global validation RMSE, và comparison chỉ dùng control stage tương ứng.

Input trên Drive:

```text
MyDrive/GeoLift_Data/teacher_subset_2000/
├── selected_2000_ids.json
├── kitti_trainval_2000.tar
├── metric_coarse_train_2000.tar
└── geometry_fused_train_2000.tar
```

## 0. Mount Drive và khai báo run

Chọn GPU runtime và đặt `AUGMENTATION_STAGE` thành `A1`, `A2` hoặc `A3`. Mỗi stage có run directory riêng; không dùng checkpoint của stage trước để tránh thay đổi initialization.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, re, io, json, shutil, tarfile, hashlib, subprocess, codecs, shlex
import numpy as np
import pandas as pd
import torch

def run_streamed(command, *, cwd=None, log_path=None, env=None):
    """Run a command with live Colab output and an optional complete UTF-8 log."""
    command = [str(part) for part in command]
    child_env = os.environ.copy()
    child_env['PYTHONUNBUFFERED'] = '1'
    if env:
        child_env.update({str(k): str(v) for k, v in env.items()})
    log_file = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_file = log_path.open('a', encoding='utf-8')
    print(f'$ {shlex.join(command)}', flush=True)
    process = subprocess.Popen(
        command, cwd=None if cwd is None else str(cwd), env=child_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
    )
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        assert process.stdout is not None
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            output = decoder.decode(chunk)
            sys.stdout.write(output)
            sys.stdout.flush()
            if log_file is not None:
                log_file.write(output)
                log_file.flush()
        output = decoder.decode(b'', final=True)
        if output:
            sys.stdout.write(output)
            sys.stdout.flush()
            if log_file is not None:
                log_file.write(output)
                log_file.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        print('\nInterrupt received; stopping child process...', flush=True)
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        if log_file is not None:
            log_file.close()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code

drive.mount('/content/drive')
GITHUB_REPO = 'https://github.com/PhuocDang2104/GeoDistill_RT.git'
LOCAL_REPO = Path('/content/GeoDistill_RT')
TAR2000_ROOT = Path('/content/drive/MyDrive/GeoLift_Data/teacher_subset_2000')
FINAL_MANIFEST = TAR2000_ROOT / 'selected_2000_ids.json'
KITTI_TAR = TAR2000_ROOT / 'kitti_trainval_2000.tar'
METRIC_TAR = TAR2000_ROOT / 'metric_coarse_train_2000.tar'
GEOMETRY_TAR = TAR2000_ROOT / 'geometry_fused_train_2000.tar'
EXPECTED_KITTI_BYTES = 2_041_970_176

AUGMENTATION_STAGE = 'A1'  # Chạy tuần tự A1 -> A2 -> A3, mỗi lần một stage.
assert AUGMENTATION_STAGE in {'A1', 'A2', 'A3'}
AUGMENTATION_PRESETS = {
    'A1': {
        'enabled': True, 'stage': 'A1', 'horizontal_flip_prob': 0.5,
        'sparse_dropout': {'enabled': False},
        'scale_jitter': {'enabled': False},
    },
    'A2': {
        'enabled': True, 'stage': 'A2', 'horizontal_flip_prob': 0.5,
        'sparse_dropout': {'enabled': True, 'min_rate': 0.0, 'max_rate': 0.2, 'min_points': 64},
        'scale_jitter': {'enabled': False},
    },
    'A3': {
        'enabled': True, 'stage': 'A3', 'horizontal_flip_prob': 0.5,
        'sparse_dropout': {'enabled': True, 'min_rate': 0.0, 'max_rate': 0.2, 'min_points': 64},
        'scale_jitter': {'enabled': True, 'min_scale': 1.0, 'max_scale': 1.1},
    },
}
AUGMENTATION = AUGMENTATION_PRESETS[AUGMENTATION_STAGE]

DATA_ROOT = Path('/content/geolift_s3_teacher_aug_data')
KITTI_ROOT = DATA_ROOT / 'kitti_bundle'
TEACHER_ROOT = DATA_ROOT / 'teacher_outputs'
SPLIT_ROOT = DATA_ROOT / 'splits'
stage_slug = AUGMENTATION_STAGE.lower()
RUN_LOCAL = Path(f'/content/geolift_s3_encnorm_teacher_kd_aug_{stage_slug}_e20')
RUN_DRIVE = Path(f'/content/drive/MyDrive/GeoLift_RT_runs/v4_s3_lite_encnorm_teacher_kd_aug_{stage_slug}_train1600_val400_e20')
CONTROL_RUNS = {
    'A1': Path('/content/drive/MyDrive/GeoLift_RT_runs/v4_s3_lite_encnorm_teacher_kd_train1600_val400_e20'),
    'A2': Path('/content/drive/MyDrive/GeoLift_RT_runs/v4_s3_lite_encnorm_teacher_kd_aug_a1_train1600_val400_e20'),
    'A3': Path('/content/drive/MyDrive/GeoLift_RT_runs/v4_s3_lite_encnorm_teacher_kd_aug_a2_train1600_val400_e20'),
}
CONTROL_RUN_DRIVE = CONTROL_RUNS[AUGMENTATION_STAGE]
EPOCHS, BATCH_SIZE, NUM_WORKERS = 20, 2, 2
FINAL_COUNT, TRAIN_COUNT, VAL_COUNT = 2000, 1600, 400
RESUME_AUG_RUN = True

if RUN_LOCAL.exists():
    shutil.rmtree(RUN_LOCAL)
for path in (DATA_ROOT, TEACHER_ROOT, SPLIT_ROOT, RUN_LOCAL, RUN_DRIVE):
    path.mkdir(parents=True, exist_ok=True)
for path in (FINAL_MANIFEST, KITTI_TAR, METRIC_TAR, GEOMETRY_TAR):
    assert path.is_file(), f'Thiếu input: {path}'
assert KITTI_TAR.stat().st_size == EXPECTED_KITTI_BYTES, KITTI_TAR.stat().st_size
assert torch.cuda.is_available(), 'Hãy chọn GPU runtime.'
payload_gib = sum(p.stat().st_size for p in (KITTI_TAR, METRIC_TAR, GEOMETRY_TAR)) / 1024**3
free_gib = shutil.disk_usage('/content').free / 1024**3
assert free_gib >= payload_gib + 12.0, (free_gib, payload_gib + 12.0)
print('GPU:', torch.cuda.get_device_name(0))
print('Augmentation stage:', AUGMENTATION_STAGE, AUGMENTATION)
print('Direct control:', CONTROL_RUN_DRIVE)
print('Run output:', RUN_DRIVE)
print(f'Local SSD {free_gib:.1f} GiB; required about {payload_gib + 12.0:.1f} GiB')

## 1. Clone source mới nhất vào Colab SSD

In [ ]:
if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
setup_log = RUN_LOCAL / 'logs' / 'notebook_setup.log'
run_streamed(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(LOCAL_REPO)], log_path=setup_log)
os.chdir(LOCAL_REPO)
requirements = LOCAL_REPO / 'requirements.txt'
if requirements.is_file():
    run_streamed([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], log_path=setup_log)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', commit)

## 2. Gate A — manifest và hai teacher TAR

Notebook bắt buộc hai TAR có đúng 2.000 canonical ID, đúng manifest và không trùng raw drive giữa train/validation.

In [ ]:
def canonical_id(member_name):
    stem = Path(member_name).stem
    old = re.match(r'^(?P<prefix>.+)_sync_image_(?P<camera>\d{2})_(?P<frame>\d{10})$', stem)
    if old:
        return f"{old.group('prefix')}_sync_image_{old.group('frame')}_image_{old.group('camera')}"
    canonical = re.match(r'^(?P<prefix>.+)_sync_image_(?P<frame>\d{10})_image_(?P<camera>\d{2})$', stem)
    if canonical:
        return stem
    raise ValueError(f'Không canonicalize được: {member_name}')

def raw_drive(sample_id):
    return sample_id.split('_sync_image_')[0]

def index_tar(path):
    result = {}
    with tarfile.open(path, 'r:*') as archive:
        for member in archive.getmembers():
            if not member.isfile() or not member.name.endswith('.npz'):
                continue
            sample_id = canonical_id(member.name)
            assert sample_id not in result, (path.name, sample_id)
            result[sample_id] = member.name
    print(path.name, 'NPZ:', len(result))
    return result

manifest = json.loads(FINAL_MANIFEST.read_text(encoding='utf-8'))
train_ids = list(manifest['train_ids'])
val_ids = list(manifest['val_ids'])
selected_ids = train_ids + val_ids
assert (len(train_ids), len(val_ids), len(set(selected_ids))) == (TRAIN_COUNT, VAL_COUNT, FINAL_COUNT)
assert not ({raw_drive(x) for x in train_ids} & {raw_drive(x) for x in val_ids})
metric_index = index_tar(METRIC_TAR)
geometry_index = index_tar(GEOMETRY_TAR)
assert set(metric_index) == set(geometry_index) == set(selected_ids)
print('Gate A passed: 2.000 matched teacher IDs; train/val raw-drive overlap = 0')

## 3. Extract teacher và KITTI bundle vào `/content`

In [ ]:
split_by_id = {x: 'train' for x in train_ids}
split_by_id.update({x: 'val' for x in val_ids})

def extract_teacher(tar_path, tar_index, role_root):
    for split in ('train', 'val'):
        (role_root / split).mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, 'r:*') as archive:
        members = {m.name: m for m in archive.getmembers()}
        ordered = sorted(((members[tar_index[x]].offset_data, x, members[tar_index[x]]) for x in selected_ids))
        for number, (_, sample_id, member) in enumerate(ordered, 1):
            source = archive.extractfile(member)
            assert source is not None
            target = role_root / split_by_id[sample_id] / f'{sample_id}.npz'
            with target.open('wb') as output:
                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            if number % 200 == 0:
                print(role_root.name, number, '/', FINAL_COUNT)

if TEACHER_ROOT.exists():
    shutil.rmtree(TEACHER_ROOT)
extract_teacher(METRIC_TAR, metric_index, TEACHER_ROOT / 'metric_coarse')
extract_teacher(GEOMETRY_TAR, geometry_index, TEACHER_ROOT / 'geometry_fused')

def safe_extract_tar(source, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with tarfile.open(source, 'r:*') as archive:
        members = archive.getmembers()
        for member in members:
            target = (destination / member.name).resolve()
            assert target == root or root in target.parents, member.name
        archive.extractall(destination, members=members)
safe_extract_tar(KITTI_TAR, KITTI_ROOT)

bundle_report = json.loads((KITTI_ROOT / 'kitti_bundle_report.json').read_text(encoding='utf-8'))
assert bundle_report['contract_ok'] is True
assert bundle_report['selected_count'] == FINAL_COUNT
bundle_manifest = json.loads((KITTI_ROOT / 'selected_2000_ids.json').read_text(encoding='utf-8'))
assert bundle_manifest['train_ids'] == manifest['train_ids']
assert bundle_manifest['val_ids'] == manifest['val_ids']
for name, expected in (('train_1600.txt', TRAIN_COUNT), ('val_400.txt', VAL_COUNT)):
    source = KITTI_ROOT / 'splits' / name
    lines = [x for x in source.read_text(encoding='utf-8').splitlines() if x.strip()]
    assert len(lines) == expected
    shutil.copy2(source, SPLIT_ROOT / name)
print('Teacher + KITTI extraction complete')

## 4. Gate B — coverage, schema và augmentation visualization

Coverage phải là 100% cho train/validation. Preview dùng transform cưỡng bức để kiểm tra RGB, sparse và teacher cùng spatial layout trước khi train; đây không phải random parameters dùng trong training.

In [ ]:
coverage = {}
for role in ('metric_coarse', 'geometry_fused'):
    coverage[role] = {}
    for split, ids in (('train', train_ids), ('val', val_ids)):
        present = {p.stem for p in (TEACHER_ROOT / role / split).glob('*.npz')}
        coverage[role][split] = len(present & set(ids)) / len(ids)
        assert present == set(ids), (role, split, len(present))

for sample_id in train_ids[:12] + val_ids[:4]:
    split = split_by_id[sample_id]
    with np.load(TEACHER_ROOT / 'metric_coarse' / split / f'{sample_id}.npz', allow_pickle=False) as data:
        assert {'D_cm', 'C_cm'} <= set(data.files)
        D_cm, C_cm = np.asarray(data['D_cm']), np.asarray(data['C_cm'])
        assert D_cm.shape == C_cm.shape and np.isfinite(D_cm).all() and np.isfinite(C_cm).all()
        assert C_cm.min() >= 0 and C_cm.max() <= 1
    with np.load(TEACHER_ROOT / 'geometry_fused' / split / f'{sample_id}.npz', allow_pickle=False) as data:
        assert {'R_G', 'C_G'} <= set(data.files)
        R_G, C_G = np.asarray(data['R_G']), np.asarray(data['C_G'])
        assert R_G.shape == C_G.shape and np.isfinite(R_G).all() and np.isfinite(C_G).all()
        assert C_G.min() >= 0 and C_G.max() <= 1
teacher_report = {'contract_ok': True, 'coverage': coverage, 'inspected_schema_samples': 16,
                  'teacher_roles': ['metric_coarse', 'geometry_fused'], 'geometry_fallback': False}
(RUN_LOCAL / 'teacher_train_val_report.json').write_text(json.dumps(teacher_report,indent=2),encoding='utf-8')
print(json.dumps(teacher_report, indent=2))

import matplotlib.pyplot as plt
from src.dataset import KITTIDepthCompletionDataset
from src.utils import make_ray_map
common_dataset_args = dict(
    data_root=KITTI_ROOT, split_root=SPLIT_ROOT, split_file='train_1600.txt', split_name='train',
    image_size=(352,1216), teacher_root=TEACHER_ROOT, load_teacher=True, load_geometry=True,
    geometry_fallback=False, load_mono=False, return_tensors=True,
)
plain_dataset = KITTIDepthCompletionDataset(**common_dataset_args)
preview_aug = json.loads(json.dumps(AUGMENTATION))
preview_aug['horizontal_flip_prob'] = 1.0
if preview_aug['sparse_dropout'].get('enabled', False):
    preview_aug['sparse_dropout'].update({'min_rate': 0.2, 'max_rate': 0.2})
if preview_aug['scale_jitter'].get('enabled', False):
    preview_aug['scale_jitter'].update({'min_scale': 1.1, 'max_scale': 1.1})
aug_dataset = KITTIDepthCompletionDataset(**common_dataset_args, augmentation=preview_aug)
plain_sample = plain_dataset[0]
np.random.seed(2026)
sample = aug_dataset[0]
assert {'D_cm','C_cm','R_G','C_G','augmentation_params'} <= set(sample) and 'D_da_raw' not in sample
assert sample['rgb'].shape == plain_sample['rgb'].shape == (3,352,1216)
assert torch.equal(sample['mask'] > 0, sample['sparse'] > 0)
assert torch.allclose(sample['ray'], torch.from_numpy(make_ray_map(sample['K'].numpy(),352,1216)), atol=1e-6)
panels = [('Plain RGB', plain_sample['rgb'].permute(1,2,0), None), ('Aug RGB', sample['rgb'].permute(1,2,0), None),
          ('Plain sparse', plain_sample['sparse'][0], 'turbo'), ('Aug sparse', sample['sparse'][0], 'turbo'),
          ('Plain D_cm', plain_sample['D_cm'][0], 'turbo'), ('Aug D_cm', sample['D_cm'][0], 'turbo'),
          ('Plain R_G', plain_sample['R_G'][0], 'turbo'), ('Aug R_G', sample['R_G'][0], 'turbo')]
fig, axes = plt.subplots(4,2,figsize=(20,14))
for ax,(title,image,cmap) in zip(axes.ravel(),panels):
    ax.imshow(image, cmap=cmap); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
print('Preview augmentation params [sx,sy,crop_x,crop_y,flip,drop_rate,points_before,points_after]:')
print(sample['augmentation_params'].tolist())

## 5. Resolve config S3 Teacher-KD + selected augmentation stage

Run bắt đầu từ cùng ImageNet pretrained initialization như control, không fine-tune từ checkpoint stage trước. Resolved config lưu nguyên augmentation preset; resume bị từ chối nếu stage/config khác.

In [ ]:
import yaml
paths_file = DATA_ROOT / f'paths_s3_teacher_kd_aug_{stage_slug}.yaml'
paths = {
    'data_root': str(KITTI_ROOT), 'split_root': str(SPLIT_ROOT),
    'train_split': 'train_1600.txt', 'val_split': 'val_400.txt', 'test_split': 'val_400.txt',
    'teacher_root': str(TEACHER_ROOT), 'student_root': str(RUN_LOCAL),
}
paths_file.write_text(yaml.safe_dump(paths, sort_keys=False), encoding='utf-8')
base_config = LOCAL_REPO / 'configs' / 'geolift_s3_lite_teacher_kd_tar2000.yaml'
assert base_config.is_file(), 'Hãy push source mới có config teacher-KD lên GitHub trước.'
cfg = yaml.safe_load(base_config.read_text(encoding='utf-8'))
cfg['paths_file'] = str(paths_file)
cfg['train']['epochs'] = EPOCHS
cfg['train']['batch_size'] = BATCH_SIZE
cfg['data']['num_workers'] = NUM_WORKERS
cfg['data']['augmentation'] = AUGMENTATION
cfg['outputs']['backup_root'] = str(RUN_DRIVE)
cfg['train']['resume'] = None
last_drive = RUN_DRIVE / 'checkpoints' / 'last.pth'
if RESUME_AUG_RUN and last_drive.is_file():
    previous_config_file = RUN_DRIVE / 'resolved_config.yaml'
    if previous_config_file.is_file():
        previous_cfg = yaml.safe_load(previous_config_file.read_text(encoding='utf-8'))
        assert previous_cfg['model'] == cfg['model'], 'Resume model config mismatch'
        assert previous_cfg['loss'] == cfg['loss'], 'Resume loss config mismatch'
        assert previous_cfg['schedule'] == cfg['schedule'], 'Resume schedule config mismatch'
        assert previous_cfg['data'].get('augmentation') == cfg['data'].get('augmentation'), 'Resume augmentation config mismatch'
        for key in ('epochs','batch_size','lr','scheduler','scheduler_total_epochs'):
            assert previous_cfg['train'][key] == cfg['train'][key], ('Resume train config mismatch',key)
    local_last = RUN_LOCAL / 'checkpoints' / 'last.pth'
    local_last.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_drive, local_last)
    best_drive = RUN_DRIVE / 'checkpoints' / 'best.pth'
    if best_drive.is_file():
        shutil.copy2(best_drive, local_last.parent / 'best.pth')
    local_logs = RUN_LOCAL / 'logs'
    local_logs.mkdir(parents=True, exist_ok=True)
    for name in (
        'train_log.csv', 'train_log.jsonl', 'train_student.log',
        'train_console.log', 'contract_tests.log', 'infer_val_console.log',
        'profile_console.log', 'artifact_sync.log',
    ):
        source = RUN_DRIVE / 'logs' / name
        if source.is_file():
            shutil.copy2(source, local_logs / name)
    cfg['train']['resume'] = str(local_last)
resolved_cfg = DATA_ROOT / f'geolift_s3_teacher_kd_aug_{stage_slug}_e20.yaml'
resolved_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
run_manifest = {
    'architecture': 'GeoLift-S3-Lite', 'experiment': f'teacher_kd_augmentation_{stage_slug}', 'epochs': EPOCHS,
    'encoder_normalize_input': cfg['model']['encoder_normalize_input'],
    'teacher_roles': ['metric_coarse:D_cm/C_cm', 'geometry_fused:R_G/C_G'],
    'metric_kd_scope': 'non_KITTI_GT_pixels_only', 'geometry_fallback': False,
    'base_objective_unchanged': True, 'augmentation_stage': AUGMENTATION_STAGE,
    'augmentation': AUGMENTATION, 'direct_control_run': str(CONTROL_RUN_DRIVE),
    'augmentation_rng': 'data_seed = global_seed + epoch + 1000003*rank; augmented workers restart each epoch',
    'scheduler_total_epochs': cfg['train']['scheduler_total_epochs'],
    'teacher_tar_bytes': {'metric': METRIC_TAR.stat().st_size, 'geometry': GEOMETRY_TAR.stat().st_size},
    'train': TRAIN_COUNT, 'val': VAL_COUNT, 'source_commit': commit,
    'config_sha256': hashlib.sha256(resolved_cfg.read_bytes()).hexdigest(),
}
artifacts = [(resolved_cfg,'resolved_config.yaml'), (paths_file,'resolved_paths.yaml'),
             (FINAL_MANIFEST,'selected_2000_ids.json')]
for source,name in artifacts:
    shutil.copy2(source, RUN_LOCAL / name)
(RUN_LOCAL / 'run_manifest.json').write_text(json.dumps(run_manifest,indent=2),encoding='utf-8')
run_streamed(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'], log_path=RUN_LOCAL / 'logs' / 'artifact_sync.log')
print(resolved_cfg.read_text(encoding='utf-8'))

## 6. Contract tests và real-batch smoke

Cell này xác nhận model vẫn có đúng 369.209 parameters, train batch chứa augmentation metadata và hai teacher role, còn validation batch tuyệt đối không augment.

In [ ]:
run_streamed(
    [sys.executable,'-m','unittest','discover','-s','tests','-v'],
    cwd=LOCAL_REPO, log_path=RUN_LOCAL / 'logs' / 'contract_tests.log',
)
from src.utils import load_project_config
from src.train_student import make_loader, preflight_teacher_coverage
from src.model_factory import build_student
from src.losses import geort_loss
import logging
loaded_cfg, loaded_paths = load_project_config(resolved_cfg)
loader = make_loader(loaded_cfg, loaded_paths, 'train', training=True)
val_loader = make_loader(loaded_cfg, loaded_paths, 'val', training=False)
preflight_teacher_coverage(loaded_cfg, loader.dataset, logging.getLogger('preflight'))
batch = next(iter(loader))
val_batch = next(iter(val_loader))
assert {'D_cm','C_cm','R_G','C_G','augmentation_params'} <= set(batch) and 'D_da_raw' not in batch
assert 'augmentation_params' not in val_batch, 'Validation must remain deterministic/plain'
assert torch.equal(batch['mask'] > 0, batch['sparse'] > 0)
aug_params = batch['augmentation_params']
assert torch.all(aug_params[:,7] <= aug_params[:,6])
if AUGMENTATION_STAGE == 'A1':
    assert torch.equal(aug_params[:,7], aug_params[:,6])
if AUGMENTATION_STAGE in {'A1','A2'}:
    assert torch.allclose(aug_params[:,0], torch.ones_like(aug_params[:,0]))
model = build_student(loaded_cfg).cuda().train()
assert sum(p.numel() for p in model.parameters()) == 369_209
gpu_batch = {k:(v[:1].cuda() if torch.is_tensor(v) else v) for k,v in batch.items()}
with torch.autocast('cuda',dtype=torch.float16):
    pred = model(*(gpu_batch[k] for k in ('rgb','sparse','mask','ray','uv','K')))
    loss0, items0 = geort_loss(pred,gpu_batch,loaded_cfg['loss'],loaded_cfg['schedule'],0,loaded_cfg['mono_ssi'])
    loss3, items3 = geort_loss(pred,gpu_batch,loaded_cfg['loss'],loaded_cfg['schedule'],3,loaded_cfg['mono_ssi'])
assert torch.isfinite(loss0) and torch.isfinite(loss3)
assert items0['w_teacher_metric'] > 0 and items0['w_teacher_geometry'] == 0
assert items3['w_teacher_geometry'] > 0 and items3['w_teacher_ordinal'] > 0
assert items0['teacher_metric_non_gt_ratio'] > 0
loss3.backward()
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
print('Augmentation batch params:', aug_params.tolist())
print('Smoke passed:', {k:items3[k] for k in items3 if k.startswith('w_teacher')})
del model,pred,batch,val_batch,gpu_batch,loader,val_loader
torch.cuda.empty_cache()

## 7. Train 20 epochs

Không nạp checkpoint control/stage trước. Nếu `RESUME_AUG_RUN=True`, notebook chỉ resume checkpoint có cùng augmentation config trong run directory của stage hiện tại.

In [ ]:
train_command = [sys.executable,'-m','src.train_student','--config',str(resolved_cfg)]
try:
    run_streamed(train_command, cwd=LOCAL_REPO, log_path=RUN_LOCAL / 'logs' / 'train_console.log')
finally:
    run_streamed(
        ['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'],
        log_path=RUN_LOCAL / 'logs' / 'artifact_sync.log',
    )
print('Training complete:', RUN_DRIVE)

## 8. Final validation inference và FP16 profile

In [ ]:
best = RUN_LOCAL / 'checkpoints' / 'best.pth'
assert best.is_file(), best
run_streamed(
    [sys.executable,'-m','src.infer_student','--config',str(resolved_cfg),'--checkpoint',str(best),'--split','val'],
    cwd=LOCAL_REPO, log_path=RUN_LOCAL / 'logs' / 'infer_val_console.log',
)
profile_out = RUN_LOCAL / 'logs' / 'geolift_s3_component_profile.json'
run_streamed(
    [sys.executable,'scripts/profile_geolift_s3.py','--config',str(resolved_cfg),'--warmup','20','--runs','100','--output',str(profile_out)],
    cwd=LOCAL_REPO, log_path=RUN_LOCAL / 'logs' / 'profile_console.log',
)
run_streamed(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'], log_path=RUN_LOCAL / 'logs' / 'artifact_sync.log')
print((RUN_LOCAL / 'logs' / 'infer_val_metrics_global.json').read_text(encoding='utf-8'))
print(profile_out.read_text(encoding='utf-8'))

## 9. So sánh stage hiện tại với direct control

A1 so với Teacher-KD plain, A2 so với A1, A3 so với A2. Gate yêu cầu cùng model/loss/schedule/split/20 epoch; khác biệt duy nhất là augmentation stage.

In [ ]:
import matplotlib.pyplot as plt
run_log = RUN_LOCAL / 'logs' / 'train_log.csv'
control_log = CONTROL_RUN_DRIVE / 'logs' / 'train_log.csv'
control_config_file = CONTROL_RUN_DRIVE / 'resolved_config.yaml'
assert run_log.is_file(), run_log
run_df = pd.read_csv(run_log)
run_df = run_df[run_df.epoch < EPOCHS].copy()
run_best = run_df.loc[run_df.val_rmse.idxmin()]
if control_log.is_file():
    assert control_config_file.is_file(), control_config_file
    control_cfg = yaml.safe_load(control_config_file.read_text(encoding='utf-8'))
    assert control_cfg['model'] == cfg['model'], 'Model mismatch'
    assert control_cfg['loss'] == cfg['loss'], 'Loss mismatch'
    assert control_cfg['schedule'] == cfg['schedule'], 'Schedule mismatch'
    assert control_cfg['data']['image_size'] == cfg['data']['image_size']
    for key in ('epochs','batch_size','lr','scheduler','scheduler_total_epochs'):
        assert control_cfg['train'][key] == cfg['train'][key], ('Train config mismatch',key)
    control_aug = control_cfg['data'].get('augmentation', {})
    control_stage = control_aug.get('stage', 'plain') if control_aug.get('enabled', False) else 'plain'
    expected_control = {'A1':'plain','A2':'A1','A3':'A2'}[AUGMENTATION_STAGE]
    assert control_stage == expected_control, (control_stage, expected_control)
    print('Direct-control model/loss/train/augmentation ladder gate passed')
    control_df = pd.read_csv(control_log)
    control_df = control_df[control_df.epoch < EPOCHS].copy()
    assert len(control_df) >= EPOCHS, f'Control chỉ có {len(control_df)} epoch trong 0..19'
    control_best = control_df.loc[control_df.val_rmse.idxmin()]
    metrics = ['val_rmse','val_mae','val_irmse','val_abs_rel','val_delta1',
               'val_rmse_0_20','val_rmse_20_40','val_rmse_40_60','val_rmse_60_80','val_rmse_80_120',
               'val_rmse_edge','val_rmse_nonedge']
    rows = []
    for metric in metrics:
        b, t = float(control_best[metric]), float(run_best[metric])
        lower_is_better = metric != 'val_delta1'
        gain = (b-t)/max(abs(b),1e-12)*100 if lower_is_better else (t-b)/max(abs(b),1e-12)*100
        rows.append({'metric':metric,'control_best':b,f'{stage_slug}_best':t,'gain_pct':gain})
    comparison = pd.DataFrame(rows)
    comparison.to_csv(RUN_LOCAL / f'augmentation_{stage_slug}_vs_control_e20.csv',index=False)
    display(comparison)
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    axes[0].plot(control_df.epoch,control_df.val_rmse,label=f'control: {expected_control}')
    axes[0].plot(run_df.epoch,run_df.val_rmse,label=f'augmentation: {AUGMENTATION_STAGE}')
    axes[0].set(xlabel='Epoch',ylabel='Validation RMSE (m)',title='Direct augmentation ablation'); axes[0].grid(); axes[0].legend()
    range_cols=['val_rmse_0_20','val_rmse_20_40','val_rmse_40_60','val_rmse_60_80']
    x=np.arange(len(range_cols)); axes[1].bar(x-.2,[control_best[c] for c in range_cols],.4,label=expected_control)
    axes[1].bar(x+.2,[run_best[c] for c in range_cols],.4,label=AUGMENTATION_STAGE)
    axes[1].set_xticks(x,['0-20','20-40','40-60','60-80']); axes[1].set(ylabel='RMSE (m)',xlabel='GT range (m)',title='Best-by-global-RMSE checkpoint'); axes[1].legend()
    fig.tight_layout(); fig.savefig(RUN_LOCAL / f'augmentation_{stage_slug}_vs_control_e20.png',dpi=180,bbox_inches='tight'); plt.show()
    print('Control best epoch:',int(control_best.epoch),'RMSE:',float(control_best.val_rmse))
else:
    print('Không tìm thấy direct-control log; hãy chạy đúng thứ tự plain -> A1 -> A2 -> A3:', control_log)
print(AUGMENTATION_STAGE, 'best epoch:',int(run_best.epoch),'RMSE:',float(run_best.val_rmse))
loss_budget = pd.DataFrame({
    'term':['S3 base','metric teacher KD','geometry SSI','geometry ordinal'],
    'weighted_value':[
        float(run_best.train_loss) - float(run_best.train_w_teacher_metric)*float(run_best.train_L_teacher_metric_kd) - float(run_best.train_w_teacher_geometry)*float(run_best.train_L_teacher_geometry_ssi) - float(run_best.train_w_teacher_ordinal)*float(run_best.train_L_teacher_geometry_ord),
        float(run_best.train_w_teacher_metric)*float(run_best.train_L_teacher_metric_kd),
        float(run_best.train_w_teacher_geometry)*float(run_best.train_L_teacher_geometry_ssi),
        float(run_best.train_w_teacher_ordinal)*float(run_best.train_L_teacher_geometry_ord),
    ]})
loss_budget.to_csv(RUN_LOCAL / f'augmentation_{stage_slug}_loss_budget.csv',index=False); display(loss_budget)
run_streamed(['rsync','-a',f'{RUN_LOCAL}/',f'{RUN_DRIVE}/'], log_path=RUN_LOCAL / 'logs' / 'artifact_sync.log')

## Drive output

```text
MyDrive/GeoLift_RT_runs/v4_s3_lite_encnorm_teacher_kd_aug_<a1|a2|a3>_train1600_val400_e20/
├── checkpoints/{best.pth,last.pth,epoch_*.pth}
├── logs/{train_log.csv,train_log.jsonl,train_student.log}
├── logs/{infer_val_metrics_global.json,geolift_s3_component_profile.json}
├── augmentation_<stage>_vs_control_e20.csv
├── augmentation_<stage>_vs_control_e20.png
├── augmentation_<stage>_loss_budget.csv
├── teacher_train_val_report.json
├── resolved_config.yaml
├── resolved_paths.yaml
└── run_manifest.json
```

Run này không tạo anonymous-test ZIP vì mục tiêu là augmentation ablation trên fixed validation. Chạy notebook ba lần theo thứ tự A1 → A2 → A3; mỗi lần chỉ đổi `AUGMENTATION_STAGE`.